# Ridge

Standard tabular ML workflow for LD50 regression.


In [12]:
from pathlib import Path
import sys
import warnings

import numpy as np

PROJECT_ROOT = Path("../..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from sklearn.linear_model import RidgeCV

from utils.modeling import artifact_paths, load_tabular_arrays, save_ml_run


In [13]:
MODEL_NAME = "ridge"
BASE_DIR = Path.cwd()

X_train, X_valid, X_test, y_train, y_valid, y_test, feature_names = load_tabular_arrays('../../data/feature_selection')
X_train = X_train.astype(np.float32)
X_valid = X_valid.astype(np.float32)
X_test = X_test.astype(np.float32)

paths = artifact_paths(
    BASE_DIR,
    MODEL_NAME,
    model_suffix=".pkl",
    include_importance=True,
)


In [14]:
try:
    from cuml.linear_model import Ridge as CumlRidge

    model = CumlRidge(alpha=1.0)
    chosen_alpha = 1.0
except Exception:
    warnings.warn("cuML Ridge not available; using sklearn RidgeCV on CPU.")
    model = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=None)

model.fit(X_train, y_train)

if hasattr(model, "alpha_"):
    chosen_alpha = float(model.alpha_)


C:\Users\Tommaso\AppData\Local\Temp\ipykernel_37000\2151313871.py:7: UserWarning: cuML Ridge not available; using sklearn RidgeCV on CPU.
  warnings.warn("cuML Ridge not available; using sklearn RidgeCV on CPU.")


In [15]:
predictions = model.predict(X_test)
metrics = save_ml_run("Ridge", model, paths, X_test, y_test, predictions, feature_names)
print(f"[Ridge] Chosen alpha: {chosen_alpha}")


[Ridge] RMSE: 0.7813 | MAE: 0.5885 | R2: 0.3167
[Ridge] Chosen alpha: 0.1
